[![Wiki](https://img.shields.io/badge/-Wiki-blue?style=flat-square&logo=github)](./wiki/Home.md)
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/neurovium/CorticalBlueprintRNN/blob/main/Model/CorticalBlueprintRNN.ipynb)
<!-- [![arXiv](https://img.shields.io/badge/arXiv-preprint-D12424?logo=arxiv)](https://arxiv.org/abs/X) -->

*The Wiki badge points to the local docs next to this notebook. For the broader project (paper + statistical analysis), see the [top-level repo README](../README.md).*

### Harnessing cortical geometry, wiring, and function as inductive biases for recurrent neural networks
Mo Shakiba, Rana Rokni, Mohammad Mohammadi, and Nima Dehghani

In [ ]:
! pip install bctpy tensorflow scipy matplotlib numpy pandas seaborn tqdm networkx --quiet # Install dependencies

In [ ]:
# @title ### `↪ DATA`
# Data lives next to this notebook:
#   - DATA/info/<session>_<scan>_<field>/   ← preprocessed arrays per field
#   - INF.md                                ← index of available fields
# (If running in Colab, clone the repo or upload DATA/ + INF.md so they sit next to the notebook.)
! cat INF.md

In [ ]:
 # @title  #### Simulation Parameters

# @markdown  - Number of simulations to run for each model variant
simulations = 1 #@param {type:"number"}
# @markdown  - **⚠️ Note:** Larger numbers will take much longer to run.

session_scan_field = '6_6_2' #@param {type:"string"}

number_of_nodes = 312 #@param {type:"number"}

network_grid = (12, 13, 2) #@param {type:"raw"}

# @markdown  - **⚠️ Note:** `number_of_nodes` and `network_grid` must match each other and the actual number of neurons in the chosen session/scan/field.

# @markdown ---

# @markdown - Apply positive sign constraint on the weights?
sign_constraint = False # @param {type:"boolean"}

# @markdown - Use only STTC for the bio weights?
use_only_sttc = False # @param {type:"boolean"}

# @markdown - Use the precision matrix for the bio weights?
use_precison_matrix = False # @param {type:"boolean"}

# @markdown ---

TASK1, TASK2, TASK3 = True, True, True  # Whether to run the task

#DEFAULT

# @markdown - Level of noise standard deviation to add to the data
noise_level = 0.05 #@param {type:"number"}

# @markdown - Regularization strength
regu_strength = 0.3 # @param {type:"slider", min:0, max:1, step:0.1}

# @markdown - EMD regularization strength
emd_strength = 0.1 # @param {type:"slider", min:0, max:1, step:0.1}

activation_function = 'relu' # Activation function to use in the network

# @markdown - Method for initializing the random network weights
random_network_initialization = 'Orthogonal' # @param ["Orthogonal", "HeNormal", "HeUniform", "GlorotNormal"]

# DO NOT CHANGE
session, scan, field = map(int, session_scan_field.split('_'))
base = f'{session}_{scan}_{field}_sttc'
if not use_only_sttc:
    base += '-precision' if use_precison_matrix else '-corr'
if sign_constraint:
    base += '_sign+'
save_folder_name = base

# @markdown ---
# @markdown  *Don't forget to execute the cell!*

#### Loading

In [ ]:
from tensorflow.keras.regularizers import Regularizer # type: ignore
from tensorflow.keras.initializers import Initializer # type: ignore
from tensorflow.python.keras import backend # type: ignore
import tensorflow.keras as keras # type: ignore
import numpy as np # type: ignore
import tensorflow as tf # type: ignore
import pandas as pd # type: ignore
import networkx as nx #type: ignore
import random #type: ignore
from networkx.algorithms.community.quality import modularity as nx_modularity #type: ignore
from tqdm import tqdm #type: ignore
from statistics import mean, stdev #type: ignore
from scipy.linalg import expm #type: ignore
from tensorflow.keras.constraints import NonNeg #type: ignore
from scipy import spatial #type: ignore
from scipy.stats import gaussian_kde #type: ignore
from scipy.stats import entropy #type: ignore
import bct #type: ignore
import os #type: ignore
import math #type: ignore

In [ ]:
connectome = pd.read_csv(f'DATA/info/{session}_{scan}_{field}/connectome_{session}_{scan}_{field}.csv', index_col=0)
connectome = connectome.values

In [ ]:
sttc = np.load(f'DATA/info/{session}_{scan}_{field}/sttc_matrix.npy', allow_pickle=True)
corr = np.load(f'DATA/info/{session}_{scan}_{field}/corr_matrix.npy', allow_pickle=True)
perc = np.load(f'DATA/info/{session}_{scan}_{field}/precision_matrix.npy', allow_pickle=True)

In [ ]:
cordinates = np.load(f'DATA/info/{session}_{scan}_{field}/positions_{session}_{scan}_{field}.npy', allow_pickle=True)
cordinates = np.array(cordinates.tolist()).reshape(3, -1)
# Normalize each axis independently
coordinates_min = cordinates.min(axis=1, keepdims=True)
coordinates_max = cordinates.max(axis=1, keepdims=True)
normalized_coordinates = (cordinates - coordinates_min) / (coordinates_max - coordinates_min)

# Convert back to list
cordinates = [normalized_coordinates[0], normalized_coordinates[1], normalized_coordinates[2]]

#### Bio Weight Init.

In [ ]:
Ws = []

# 1. Ensure symmetry in STTC matrix
i_lower = np.tril_indices(sttc.shape[0], -1)
sttc[i_lower[1], i_lower[0]] = sttc[i_lower]

# # 2. Print data properties for inspection
# print(f"STTC range: {np.min(sttc):.4f} to {np.max(sttc):.4f}, mean: {np.mean(sttc):.4f}")
# print(f"Corr range: {np.min(corr):.4f} to {np.max(corr):.4f}, mean: {np.mean(corr):.4f}")

# 3. Normalize corr and sttc with min-max scaling
def min_max_scale(x):
    min_x, max_x = np.min(x), np.max(x)
    if max_x == min_x:  # Avoid division by zero
        return np.ones_like(x)
    return (x - min_x) / (max_x - min_x)

normalized_corr = min_max_scale(corr)
normalized_sttc = min_max_scale(sttc)
normalized_perc = min_max_scale(perc)

for _ in range(simulations):
    # 4. Initialize weights with adjusted lognormal parameters
    sigma = 0.5
    mu = -0.5  # Adjusted for a higher mean
    lognormal_weights = np.random.lognormal(mu, sigma, size=corr.shape)

    # 5. Combine weights with biological constraints
    if use_only_sttc:
        bio_weights = lognormal_weights * normalized_sttc
    else:
        if use_precison_matrix:
            bio_weights = lognormal_weights * normalized_perc * normalized_sttc
        else:
            # Original idea
            bio_weights = lognormal_weights * normalized_corr * normalized_sttc

    # 6. Scale to a desired mean weight
    desired_mean = 0.1
    current_mean = np.mean(bio_weights)
    scaling_factor = desired_mean / current_mean if current_mean > 0 else 1.0
    bio_weights *= scaling_factor

    # 7. Apply minimum threshold
    min_threshold = 0.01
    bio_weights = np.maximum(bio_weights, min_threshold * (bio_weights != 0))

    # 8. Control spectral radius for stability
    eigenvalues = np.linalg.eigvals(bio_weights)
    spectral_radius = np.max(np.abs(eigenvalues))
    if spectral_radius > 0:
        bio_weights *= 0.95 / spectral_radius

    # 9. Convert to tensor
    W_bio = tf.convert_to_tensor(bio_weights, dtype=tf.float32)

    Ws.append(W_bio)

#### Tasks

##### 1-Step Inference \ Go/NoGo \ PerceptualDecisionMaking

In [ ]:
class mazeGeneratorI():
    '''
    Objects of the mazeGeneratorI class can create numpy and tf datasets of the first choice of the maze task.
    Task structure:
        Goal presentation, followed by delay period, followed by choice options.
    Response:
        One response required from agent at end of episode. Direction (Left, Up, Right, Down) of first step.
    Encoding:
        Both observations and labels are OneHot encoded.
    Usage:
        The two only function a user should need to access are "construct_numpy_data" and "construct_tf_data"
    Options:
        Both data construction methods have an option to shuffle the labels of data.
        The numpy data construction method allows to also return the maze identifiers.
    '''
    def __init__(self, goal_presentation_steps, delay_steps, choices_presentation_steps):
        self.version = 'v1.2.0'

        # Import variables defining episode
        self.goal_presentation_steps = goal_presentation_steps
        self.delay_steps = delay_steps
        self.choices_presentation_steps = choices_presentation_steps

        # Construct mazes dataframe
        ## Add encoded versions of the goal / choices presentations and the next step response
        self.mazesdf = self.import_maze_dic()
        self.mazesdf['Goal_Presentation'] = self.mazesdf['goal'].map({
            7:np.concatenate((np.array([1,0,0,0]),np.repeat(0,4))),
            9:np.concatenate((np.array([0,1,0,0]),np.repeat(0,4))),
            17:np.concatenate((np.array([0,0,1,0]),np.repeat(0,4))),
            19:np.concatenate((np.array([0,0,0,1]),np.repeat(0,4)))})
        self.mazesdf['Choices_Presentation']=self.mazesdf['ChoicesCategory'].map(lambda x: self.encode_choices(x=x))
        self.mazesdf['Step_Encoded']=self.mazesdf['NextFPmap'].map(lambda x: self.encode_next_step(x=x))

    def construct_numpy_data(self, number_of_problems, return_maze_identifiers = False, np_shuffle_data = False):
        # Create a new column which hold the vector for each problem
        self.mazesdf['Problem_Vec']=self.mazesdf.apply(lambda x: self.create_problem_observation(row= x,goal_presentation_steps= self.goal_presentation_steps,delay_steps= self.delay_steps,choices_presentation_steps= self.choices_presentation_steps), axis=1)
        # Set a random order of maze problems for the current session
        self.mazes_order = np.random.randint(0,8,number_of_problems)

        # Create vectors, holding observations and labels
        session_observation =np.array([])
        session_labels = np.array([])
        for i in self.mazes_order:
            session_observation = np.append(session_observation,self.mazesdf.iloc[i]['Problem_Vec'])
            session_labels = np.append(session_labels,self.mazesdf.iloc[i]['Step_Encoded'])

        # Reshape vectors to fit network observation and response space
        session_length = self.goal_presentation_steps + self.delay_steps + self.choices_presentation_steps
        session_observation = np.reshape(session_observation, (-1,session_length,8)).astype('float32')
        session_labels = np.reshape(session_labels, (-1,4)).astype('float32')

        # If np_shuffle_data == 'Labels, the order of labels is shuffled to randomise correct answers
        if np_shuffle_data == 'Labels':
          shuffle_generator = np.random.default_rng(38446)
          shuffle_generator.shuffle(session_labels,axis=0)

        # If return_maze_identifiers == 'IDs', return the array with maze IDs alongside the regular returns (observations, labels)
        if return_maze_identifiers == 'IDs':
          return session_observation, session_labels, self.mazes_order

        return session_observation, session_labels

    def construct_tf_data(self, number_of_problems, batch_size, tf_shuffle_data = False):
        # Create dataset as described by numpy dataset function and transform it into a TF dataset
        npds, np_labels = self.construct_numpy_data(number_of_problems=number_of_problems, np_shuffle_data = tf_shuffle_data)
        tfdf = tf.data.Dataset.from_tensor_slices((npds, np_labels))
        tfdf = tfdf.batch(batch_size)
        return tfdf

    def reset_construction_params(self, goal_presentation_steps, delay_steps, choices_presentation_steps):
        self.goal_presentation_steps = goal_presentation_steps
        self.delay_steps = delay_steps
        self.choices_presentation_steps = choices_presentation_steps

    def import_maze_dic(self, mazeDic=None):
        if mazeDic == None:
            # Set up dataframe with first choices of maze task
            ## The dictionary was generated using MazeMetadata.py (v1.0.0) and the following call:
            ### mazes.loc[(mazes['Nsteps']==2)&(mazes['ChoiceNo']=='ChoiceI')][['goal','ChoicesCategory','NextFPmap']].reset_index(drop=True).to_dict()
            self.mazesDic = {'goal': {0: 9, 1: 9, 2: 19, 3: 17, 4: 17, 5: 7, 6: 19, 7: 7},
            'ChoicesCategory': {0: 'ul',
            1: 'rd',
            2: 'ld',
            3: 'rd',
            4: 'ul',
            5: 'ur',
            6: 'lr',
            7: 'lr'},
            'NextFPmap': {0: 'u', 1: 'r', 2: 'd', 3: 'd', 4: 'l', 5: 'u', 6: 'r', 7: 'l'}}
        else:
            self.mazesDic = mazeDic

        # Create and return dataframe
        return pd.DataFrame(self.mazesDic)

    def encode_choices(self, x):
        # Helper function to create the observation vector for choice periods
        choices_sec = np.repeat(0,4)
        choicesEncoding = pd.Series(list(x))
        choicesEncoding = choicesEncoding.map({'l':1,'u':2,'r':3,'d':4})
        for encodedChoice in choicesEncoding:
            choices_sec[encodedChoice-1]=1
        return np.concatenate((np.repeat(0,4),choices_sec))

    def encode_next_step(self, x):
        # Helper function to change the response / action to a OneHot encoded vector
        step_sec = np.repeat(0,4)
        stepEncoding = pd.Series(list(x))
        stepEncoding = stepEncoding.map({'l':1,'u':2,'r':3,'d':4})
        for encodedStep in stepEncoding:
            step_sec[encodedStep-1]=1
        return step_sec

    def create_problem_observation(self, row, goal_presentation_steps, delay_steps, choices_presentation_steps):
        # Helper function to create one vector describing the entire outline of one maze problem (Goal presentation, Delay Period, and Choices Presentation)
        goal_vec = np.tile(row['Goal_Presentation'], goal_presentation_steps)
        delay_vec = np.tile(np.repeat(0,8), delay_steps)
        choices_vec = np.tile(row['Choices_Presentation'], choices_presentation_steps)
        problem_vec = np.concatenate((goal_vec,delay_vec,choices_vec))
        return problem_vec

    def __repr__(self):
        return '\n'.join([
            f'Maze DataSet Generator',
            f'Goal Presentation Steps: {self.goal_presentation_steps}',
            f'Delay Steps: {self.delay_steps}',
            f'Choices Presentation Steps: {self.choices_presentation_steps}'])


In [ ]:
class GoNogoGeneratorI:
    '''
    Objects of the GoNogoGeneratorI class generate Go/NoGo task datasets in numpy and TensorFlow formats.

    Task:
        Fixation → Stimulus → Delay → Decision.
        One response required at decision time: Go or NoGo.

    Encoding:
        Observations and labels are one-hot encoded.

    Main Methods:
        - construct_numpy_data: Returns numpy arrays.
        - construct_tf_data: Returns batched tf.data.Dataset.

    Options:
        - shuffle_labels (in construct_numpy_data): Optional label shuffling.
        - return_trial_identifiers: Optionally returns trial indices.
    '''

    def __init__(self, fixation_steps, stimulus_steps, delay_steps, decision_steps):
        self.version = 'v1.0.1'

        self.fixation_steps = fixation_steps
        self.stimulus_steps = stimulus_steps
        self.delay_steps = delay_steps
        self.decision_steps = decision_steps

        self.trialsdf = self._create_trials_df()

        # Precompute encodings
        fixation_array = np.array([1, 0, 0])
        self.trialsdf['Fixation_Presentation'] = [fixation_array for _ in range(len(self.trialsdf))]
        self.trialsdf['Stimulus_Presentation'] = self.trialsdf['ground_truth'].apply(
            lambda x: np.array([0, 1, 0]) if x == 0 else np.array([0, 0, 1])
        )
        self.trialsdf['Decision_Encoding'] = self.trialsdf['ground_truth'].apply(
            lambda x: np.array([1, 0]) if x == 0 else np.array([0, 1])
        )

    def _create_trials_df(self):
        # Expandable to more trial types in future
        trials_dict = {
            'trial_id': [0, 1],
            'ground_truth': [0, 1],  # 0: NoGo, 1: Go
            'description': ['NoGo', 'Go']
        }
        return pd.DataFrame(trials_dict)

    def _create_problem_observation(self, row):
        # Build the full observation sequence for one trial
        fixation_vec = np.tile(row['Fixation_Presentation'], (self.fixation_steps, 1))
        stimulus_vec = np.tile(row['Stimulus_Presentation'], (self.stimulus_steps, 1))
        delay_vec = np.tile(row['Fixation_Presentation'], (self.delay_steps, 1))
        decision_vec = np.tile(np.array([0, 0, 0]), (self.decision_steps, 1))

        return np.concatenate((fixation_vec, stimulus_vec, delay_vec, decision_vec), axis=0)

    def construct_numpy_data(self, number_of_problems, return_trial_identifiers=False, shuffle_labels=False):
        # Random trial selection
        trials_order = np.random.randint(0, len(self.trialsdf), number_of_problems)
        observations = []
        labels = []

        for i in trials_order:
            row = self.trialsdf.iloc[i]
            obs = self._create_problem_observation(row)
            observations.append(obs)
            labels.append(row['Decision_Encoding'])

        observations = np.array(observations, dtype='float32')  # (B, T, 3)
        labels = np.array(labels, dtype='float32')              # (B, 2)

        if shuffle_labels:
            combined = list(zip(observations, labels))
            random.shuffle(combined)
            observations, labels = zip(*combined)
            observations = np.array(observations, dtype='float32')
            labels = np.array(labels, dtype='float32')

        if return_trial_identifiers:
            return observations, labels, trials_order
        return observations, labels

    def construct_tf_data(self, number_of_problems, batch_size, shuffle=False):
        np_obs, np_labels = self.construct_numpy_data(
            number_of_problems=number_of_problems, shuffle_labels=shuffle
        )
        tfds = tf.data.Dataset.from_tensor_slices((np_obs, np_labels))
        return tfds.batch(batch_size)

    def reset_construction_params(self, fixation_steps, stimulus_steps, delay_steps, decision_steps):
        self.fixation_steps = fixation_steps
        self.stimulus_steps = stimulus_steps
        self.delay_steps = delay_steps
        self.decision_steps = decision_steps

    def __repr__(self):
        return '\n'.join([
            f'Go/Nogo Task Generator (v{self.version})',
            f'Fixation Steps: {self.fixation_steps}',
            f'Stimulus Steps: {self.stimulus_steps}',
            f'Delay Steps: {self.delay_steps}',
            f'Decision Steps: {self.decision_steps}'
        ])

In [ ]:
class PerceptualDecisionMakingGeneratorI:
    '''
    Dataset generator for the Perceptual Decision Making task.

    Task structure:
        Fixation → Stimulus → Delay.
        Response: one choice (0 or 1) at end of sequence.

    Features:
        - One-hot encoded labels
        - Optional trial shuffling
        - Stimulus encoding via cosine similarity on ring

    Usage:
        Use `construct_numpy_data()` or `construct_tf_data()`.
    '''

    def __init__(self, dt=100, timing=None, cohs=None, sigma=1.0, dim_ring=2, seed=None):
        # Initialize time step
        self.dt = dt

        # Timing defaults
        default_timing = {'fixation': 100, 'stimulus': 2000, 'delay': 0, 'decision': 100}
        if timing is None:
            self.timing = default_timing
        else:
            self.timing = {k: timing.get(k, default_timing[k]) for k in default_timing}

        # Task-specific params
        self.cohs = np.array(cohs if cohs is not None else [0, 6.4, 12.8, 25.6, 51.2])
        self.sigma = sigma / np.sqrt(self.dt)  # Scale noise by time step
        self.dim_ring = dim_ring
        self.theta = np.linspace(0, 2 * np.pi, self.dim_ring, endpoint=False)

        # Random number generator for reproducibility
        self.rng = np.random.default_rng(seed)

        # Precompute steps
        self.fixation_steps = self.timing['fixation'] // self.dt
        self.stimulus_steps = self.timing['stimulus'] // self.dt
        self.delay_steps = self.timing['delay'] // self.dt
        self.total_steps = self.fixation_steps + self.stimulus_steps + self.delay_steps

    def generate_trial(self):
        """Generate a single observation sequence and label."""
        # Choose random coherence and correct direction
        coh = self.rng.choice(self.cohs)
        ground_truth = self.rng.choice(self.dim_ring)
        stim_theta = self.theta[ground_truth]

        # Create stimulus pattern
        stim_pattern = np.cos(self.theta - stim_theta) * (coh / 200) + 0.5  # Evidence signal

        obs_sequence = []
        for t in range(self.total_steps):
            if t < self.fixation_steps:
                obs = [1.0] + [0.0] * self.dim_ring
            elif t < self.fixation_steps + self.stimulus_steps:
                noise = self.rng.normal(0, self.sigma, size=self.dim_ring)
                obs = [1.0] + (stim_pattern + noise).tolist()
            else:
                obs = [1.0] + [0.0] * self.dim_ring
            obs_sequence.append(obs)

        # One-hot label
        label = np.zeros(self.dim_ring)
        label[ground_truth] = 1.0

        return np.array(obs_sequence, dtype='float32'), label.astype('float32')

    def construct_numpy_data(self, number_of_trials, shuffle_trials=False):
        """Build a dataset as NumPy arrays."""
        data = [self.generate_trial() for _ in range(number_of_trials)]

        if shuffle_trials:
            self.rng.shuffle(data)

        observations, labels = zip(*data)
        return np.array(observations), np.array(labels)

    def construct_tf_data(self, number_of_trials, batch_size, shuffle_trials=False):
        """Return batched tf.data.Dataset."""
        obs, labels = self.construct_numpy_data(number_of_trials, shuffle_trials=shuffle_trials)
        dataset = tf.data.Dataset.from_tensor_slices((obs, labels))
        return dataset.batch(batch_size)

    def __repr__(self):
        return '\n'.join([
            'PerceptualDecisionMaking Dataset Generator',
            f'  dt: {self.dt} ms',
            f'  timing: {self.timing}',
            f'  coherence levels: {self.cohs}',
            f'  sigma: {self.sigma:.4f}',
            f'  choices: {self.dim_ring} directions'
        ])

##### Generate datasets for training

**TASK 1** / 1-step Inference

In [ ]:
if TASK1:
    gen1 = mazeGeneratorI(goal_presentation_steps=20,delay_steps=10,choices_presentation_steps=20)
    tfdf1 = gen1.construct_tf_data(number_of_problems=5120, batch_size=128)
    tfdf_test1 = gen1.construct_tf_data(number_of_problems=2560, batch_size=128)
    tfdf_val1 = gen1.construct_tf_data(number_of_problems=2560, batch_size=128)
    example_data1 = next(iter(tfdf1))

**TASK 2** / GoNogo

In [ ]:
if TASK2:
    gen2 = GoNogoGeneratorI(fixation_steps=5, stimulus_steps=20, delay_steps=10, decision_steps=5)
    tfdf2 = gen2.construct_tf_data(number_of_problems=5120, batch_size=128)
    tfdf_test2 = gen2.construct_tf_data(number_of_problems=2560, batch_size=128)
    tfdf_val2 = gen2.construct_tf_data(number_of_problems=2560, batch_size=128)
    example_data2 = next(iter(tfdf2))

**TASK 3** / PerceptualDecisionMaking

In [ ]:
# dt=100: Time step of 100 ms.
# timing: Defines periods in milliseconds, resulting in 1 step fixation, 20 steps stimulus, and 10 steps delay (31 steps total).
# cohs: Five coherence levels for varying difficulty.
# sigma=1.0: Noise parameter.
# dim_ring=2: Binary choice task.

if TASK3:
    # Initialize the generator
    gen3 = PerceptualDecisionMakingGeneratorI(
        dt=100,
        timing={'fixation': 100, 'stimulus': 3000, 'delay': 1000},
        cohs=[0, 6.4, 12.8, 25.6, 51.2],
        sigma=1.0,
        dim_ring=2
    )
    tfdf3 = gen3.construct_tf_data(number_of_trials=5120, batch_size=128)
    tfdf_test3 = gen3.construct_tf_data(number_of_trials=2560, batch_size=128)
    tfdf_val3 = gen3.construct_tf_data(number_of_trials=2560, batch_size=128)
    example_data3 = next(iter(tfdf3))

#### Regularization

In [ ]:
class ConnectomeInitializer(Initializer):
    def __init__(self, connectome_matrix):
        self.connectome_matrix = connectome_matrix

    def __call__(self, shape, dtype=None):
        return self.connectome_matrix

In [ ]:
class W(Regularizer):
  def __init__(self, w=0.01):
    # Set w regularisation strength to default of 0.01 if no value given
    w = 0.01 if w is None else w
    self._check_penalty_number(w)

    # Transform regularisation strength to TF's standard float format
    self.w = backend.cast_to_floatx(w)

  def __call__(self, x):
    # Add calculation of loss here.
    # L1 for reference: self.l1 * math_ops.reduce_sum(math_ops.abs(x))
    abs_weight_matrix = tf.math.abs(x)

    w_loss = tf.math.reduce_sum(abs_weight_matrix)

    return w_loss

  def _check_penalty_number(self, x):
    """check penalty number availability, raise ValueError if failed."""
    if not isinstance(x, (float, int)):
      raise ValueError(('Value: {} is not a valid regularization penalty number, '
                        'expected an int or float value').format(x))

  def get_config(self):
    return {'w': float(self.w)}

In [ ]:
class WD(Regularizer):
  def __init__(self, wd=0.01, neuron_num = 100, network_structure = (5,5,4), coordinates_list = None, distance_power = 1, distance_metric = 'euclidean'):
    self.distance_power = distance_power

    # Set wd regularisation strength to default of 0.01 if no value given
    wd = 0.01 if wd is None else wd
    self._check_penalty_number(wd)

    # Transform regularisation strength to TF's standard float format
    self.wd = backend.cast_to_floatx(wd)

    # Set up tensor with distance matrix
    ## Set up neurons per dimension
    nx = np.arange(network_structure[0])
    ny = np.arange(network_structure[1])
    nz = np.arange(network_structure[2])

    ## Set up coordinate grid
    [x,y,z] = np.meshgrid(nx,ny,nz)
    self.coordinates = [x.ravel(),y.ravel(),z.ravel()]

    ## Override coordinate grid if one if provided in init
    if coordinates_list!=None:
      self.coordinates = coordinates_list

    ## Check neuron number / number of coordinates
    if (len(self.coordinates[0])==neuron_num)&(len(self.coordinates[1])==neuron_num)&(len(self.coordinates[2])==neuron_num):
      pass
    else:
      raise ValueError('Network / coordinate structure does not match the number of neurons.')

    ## Calculate the euclidean distance matrix
    euclidean_vector = spatial.distance.pdist(np.transpose(self.coordinates), metric=distance_metric)
    euclidean = spatial.distance.squareform(euclidean_vector**self.distance_power)
    self.distance_matrix = euclidean.astype('float32')
    self.spatial_cost_matrix = self.distance_matrix

    ## Add minimal cost for recurrent self connection (on diagonal)
    # diag_dist = np.diag(np.repeat(0.1,neuron_num)).astype('float32')
    # self.distance_matrix = self.distance_matrix + diag_dist

    ## Create tensor from distance matrix
    self.distance_tensor =  tf.convert_to_tensor(self.distance_matrix)

  def __call__(self, x):
    # Add calculation of loss here.
    # L1 for reference: self.l1 * math_ops.reduce_sum(math_ops.abs(x))
    abs_weight_matrix = tf.math.abs(x)

    #wd_loss = self.wd * tf.math.multiply(abs_weight_matrix, self.distance_tensor)
    #wd_loss = tf.math.reduce_sum(abs_weight_matrix)
    wd_loss = self.wd * tf.math.reduce_sum(tf.math.multiply(abs_weight_matrix, self.distance_tensor))

    return wd_loss

  def _check_penalty_number(self, x):
    """check penalty number availability, raise ValueError if failed."""
    if not isinstance(x, (float, int)):
      raise ValueError(('Value: {} is not a valid regularization penalty number, '
                        'expected an int or float value').format(x))

  def get_config(self):
    return {'wd': float(self.wd)}

In [ ]:
class WDC(WD):
    def __init__(self, wd=0.01, comms_factor = 1, neuron_num = 100, network_structure = (5,5,4), coordinates_list = None, distance_power = 1, distance_metric = 'euclidean'):
      WD.__init__(self, wd, neuron_num , network_structure , coordinates_list, distance_power , distance_metric)
      self.comms_factor = comms_factor

    def __call__(self, x):
      # Take absolute of weights
      abs_weight_matrix = tf.math.abs(x)

      # Calulcate weighted communicability (see reference in docstring)
      stepI = tf.math.reduce_sum(abs_weight_matrix, axis=1)
      stepII = tf.math.pow(stepI, -0.5)
      stepIII = tf.linalg.diag(stepII)
      stepIV = tf.linalg.expm(stepIII@abs_weight_matrix@stepIII)
      comms_matrix = tf.linalg.set_diag(stepIV, tf.zeros(stepIV.shape[0:-1]))

      # Multiply absolute weights with communicability weights
      comms_matrix = comms_matrix**self.comms_factor
      comms_weight_matrix = tf.math.multiply(abs_weight_matrix, comms_matrix)

      # Multiply comms weights matrix with distance tensor, scale the mean, and return as loss
      wdc_loss = self.wd * tf.math.reduce_sum(tf.math.multiply(comms_weight_matrix , self.distance_tensor))

      return wdc_loss

In [ ]:
class WDC_EMD(WD):
    def __init__(self, wd=0.01, emd_factor=0.1, neuron_num=100,
                 network_structure=(5,5,4), coordinates_list=None,
                 distance_power=1, distance_metric='euclidean',
                 empirical_connectome=None):
        WD.__init__(self, wd, neuron_num, network_structure, coordinates_list,
                    distance_power, distance_metric)
        self.emd_factor = emd_factor

        # Process empirical connectome to get empirical communicability
        if empirical_connectome is not None:
            self.empirical_comms = self._calculate_empirical_communicability(empirical_connectome)
        else:
            raise ValueError("Empirical connectome must be provided for EMD calculation")

    def _calculate_empirical_communicability(self, connectome):
        """Calculate weighted communicability matrix from empirical connectome with proper handling of zeros"""
        # Take absolute of connectome weights to ensure positivity
        abs_connectome = np.abs(connectome)

        # Calculate weighted communicability using the method from Crofts & Higham
        stepI = np.sum(abs_connectome, axis=1)

        # Handle zeros to avoid division by zero
        epsilon = 1e-10
        stepI = np.where(stepI == 0, epsilon, stepI)

        stepII = np.power(stepI, -0.5)
        stepIII = np.diag(stepII)

        # Use scipy's expm for numpy arrays
        stepIV = expm(stepIII @ abs_connectome @ stepIII)

        # Set diagonal to zero
        np.fill_diagonal(stepIV, 0)

        # Flatten the communicability matrix into a vector for EMD calculation
        comms_flat = stepIV.flatten()

        return tf.convert_to_tensor(comms_flat, dtype=tf.float32)

    def _calculate_artificial_communicability(self, weight_matrix):
        """Calculate weighted communicability matrix from network weights with robust zero handling"""
        # Take absolute of weights
        abs_weight_matrix = tf.math.abs(weight_matrix)

        # Calculate weighted communicability
        stepI = tf.math.reduce_sum(abs_weight_matrix, axis=1)

        # Add small epsilon to avoid division by zero
        epsilon = 1e-10
        stepI = stepI + epsilon

        stepII = tf.math.pow(stepI, -0.5)
        stepIII = tf.linalg.diag(stepII)
        stepIV = tf.linalg.expm(stepIII @ abs_weight_matrix @ stepIII)
        comms_matrix = tf.linalg.set_diag(stepIV, tf.zeros(stepIV.shape[0:-1]))

        # Flatten the communicability matrix for EMD calculation
        comms_flat = tf.reshape(comms_matrix, [-1])

        return comms_flat

    def _calculate_emd(self, u_values, v_values):
        """
        Calculate Earth Mover's Distance (Wasserstein-1) between two distributions.
        Implementation based on the 1D Wasserstein distance formula using sorted values.
        """
        # Sort both distributions
        u_sorted = tf.sort(u_values)
        v_sorted = tf.sort(v_values)

        # Calculate the absolute differences between corresponding quantiles
        # This is the 1D formula for Wasserstein distance
        diff = tf.abs(u_sorted - v_sorted)

        # Sum the differences (equivalent to the integral in the EMD formula)
        emd = tf.reduce_mean(diff)

        return emd

    def __call__(self, x):
        # Take absolute of weights
        abs_weight_matrix = tf.math.abs(x)

        # Calculate spatial loss (original wd formulation)
        spatial_loss = self.wd * tf.math.reduce_sum(
            tf.math.multiply(abs_weight_matrix, self.distance_tensor))

        # Calculate artificial communicability
        artificial_comms = self._calculate_artificial_communicability(x)

        # Calculate EMD between empirical and artificial communicability distributions
        emd_loss = self._calculate_emd(self.empirical_comms, artificial_comms)

        # Total loss combines spatial embedding and EMD loss
        total_loss = spatial_loss + self.emd_factor * emd_loss

        return total_loss

#### Model Defenition

In [ ]:
class RNNWeightMatrixHistoryI(keras.callbacks.Callback):
    '''
    Saves the RNN_Weight_Matrix to the training history before
    the start of training and after finishing each epoch.

    The network model needs to be build manually before calling fit() method
    for this callback to work.
    '''
    def __init__(self, RNN_layer_number = 0):
        super(RNNWeightMatrixHistoryI, self).__init__()
        self.RNN_layer_number = RNN_layer_number

    def on_train_begin(self, logs=None):
        # Create key for RNN_Weight_Matrix in history
        self.model.history.history['RNN_Weight_Matrix'] = []
        #print("Created key for RNN_Weight_Matrix in history.")

        # Save matrix before start of training
        self.model.history.history['RNN_Weight_Matrix'].append(self.model.layers[self.RNN_layer_number].get_weights()[1])
        #print("Saved RNN_Weight_Matrix to history.")

    def on_epoch_end(self, epoch, logs=None):
        # Save RNN_Weight_Matrix to history
        self.model.history.history['RNN_Weight_Matrix'].append(self.model.layers[self.RNN_layer_number].get_weights()[1])
        #print("Saved RNN_Weight_Matrix to history.")

In [ ]:
tf.keras.backend.clear_session()
regu1 = WDC(wd=regu_strength, coordinates_list=cordinates, neuron_num=number_of_nodes, distance_power=1) #W*D*C
regu2 = WDC(wd=regu_strength, coordinates_list=cordinates, neuron_num=number_of_nodes, distance_power=1) #WD*C
regu3 = WDC(wd=regu_strength, neuron_num=number_of_nodes, distance_power=1, network_structure=network_grid) #WDC
regu4 = WD(wd=regu_strength, coordinates_list=cordinates, neuron_num=number_of_nodes, distance_power=1) #WD*
regu5 = WD(wd=regu_strength, neuron_num=number_of_nodes, distance_power=1, network_structure=network_grid) #WD
regu6 = W(w=regu_strength) #W
regu7 = WDC_EMD(wd=regu_strength, coordinates_list=cordinates, neuron_num=number_of_nodes, distance_power=1, empirical_connectome=connectome, emd_factor=emd_strength) #W*D*C*
regu8 = WDC_EMD(wd=regu_strength, coordinates_list=cordinates, neuron_num=number_of_nodes, distance_power=1, empirical_connectome=connectome, emd_factor=emd_strength) #WD*C*
regu9 = WDC_EMD(wd=regu_strength, neuron_num=number_of_nodes, network_structure=network_grid, distance_power=1, empirical_connectome=connectome, emd_factor=emd_strength) #W*DC*

# Permutations
regu10 = WDC(wd=regu_strength, coordinates_list=cordinates, neuron_num=number_of_nodes, distance_power=1) #W!D*C
regu11 = WDC_EMD(wd=regu_strength, coordinates_list=cordinates, neuron_num=number_of_nodes, distance_power=1, empirical_connectome=connectome, emd_factor=emd_strength) #W!D*C*

In [ ]:
regularizers = {'W*D*C': regu1, 'WD*C': regu2 , 'WDC': regu3, 'WD*': regu4, 'WD': regu5, 'W': regu6,
                'W*D*C*': regu7, 'WD*C*': regu8 ,'W*DC*': regu9,
                'W!D*C': regu10, 'W!D*C*': regu11}

#### Model Training

In [ ]:
def model_metrics(hist):
        acc = hist.history['accuracy'][-1]
        l0ss = hist.history['loss'][-1]
        val_acc = hist.history['val_accuracy'][-1]
        val_l0ss = hist.history['val_loss'][-1]

        # Get RAW recurrent weights from callback history
        RNN_weight_matrix_raw = hist.history['RNN_Weight_Matrix'][-1]
        RNN_weight_matrix = np.abs(RNN_weight_matrix_raw)  # For graph operations

        if np.isinf(RNN_weight_matrix).any() or np.isnan(RNN_weight_matrix).any():
            return acc, l0ss, val_acc, val_l0ss, np.nan, np.nan, np.nan, np.nan

        # --- Entropy of Weight Distribution ---
        weights_flat = RNN_weight_matrix_raw.flatten()
        kde = gaussian_kde(weights_flat)
        # Evaluate KDE on a grid
        grid = np.linspace(weights_flat.min(), weights_flat.max(), 100)
        probs = kde(grid)
        probs /= probs.sum()  # Normalize to sum to 1
        entropy_weight = entropy(probs, base=2)

        # --- Graph Analysis ---
        thresh = np.quantile(RNN_weight_matrix, q=0.9)
        binary_weight_matrix = (RNN_weight_matrix > thresh).astype(int)
        G = nx.from_numpy_array(binary_weight_matrix)
        G.remove_edges_from(nx.selfloop_edges(G))

        # Compute modularity
        communities = list(nx.community.greedy_modularity_communities(G))
        q_stat = nx_modularity(G, communities)

        #Compute assortativity
        degree_assortativity = nx.degree_assortativity_coefficient(G)

        # small-worldness calculation
        A = nx.to_numpy_array(G)
        nodes = A.shape[0]
        clu = np.mean(bct.clustering_coef_bu(A))
        pth = bct.efficiency_bin(A)
        # Run nperm null models
        nperm = 1000
        cluperm = np.zeros((nperm,1))
        pthperm = np.zeros((nperm,1))
        for perm in range(nperm):
            Wperm = np.random.rand(nodes,nodes)
            # Make it into a matrix
            Wperm = np.matrix(Wperm)
            # Make symmetrical
            Wperm = Wperm+Wperm.T
            Wperm = np.divide(Wperm,2)
            # Binarise
            threshold, upper, lower = .7,1,0
            Aperm = np.where(Wperm>threshold,upper,lower)
            # Take null model
            cluperm[perm] = np.mean(bct.clustering_coef_bu(Aperm))
            pthperm[perm] = bct.efficiency_bin(Aperm)
        # Take the average of the nulls
        clunull = np.mean(cluperm)
        pthnull = np.mean(pthperm)
        # Calculate small-worldness
        smw = np.divide(np.divide(clu,clunull),np.divide(pthnull, pth))

        return acc, l0ss, val_acc, val_l0ss, q_stat, smw, degree_assortativity, entropy_weight

##### **TASK1**

In [ ]:
Accuracy1 = {key: [] for key in regularizers.keys()}
Loss1 = {key: [] for key in regularizers.keys()}
Validation_Accuracy1 = {key: [] for key in regularizers.keys()}
Validation_Loss1 = {key: [] for key in regularizers.keys()}
Modularity1 = {key: [] for key in regularizers.keys()}
SmallWorldness1 = {key: [] for key in regularizers.keys()}
Assortativity1 = {key: [] for key in regularizers.keys()}
Entropy1 = {key: [] for key in regularizers.keys()}

In [ ]:
if TASK1:
    MODELS1 = []
    for i in range(simulations):
        model = {}
        for key, regu in regularizers.items():
            if 'W*' in key:
                initializer = ConnectomeInitializer(Ws[i])

            elif 'W!' in key: #Permutation
                # Generate a random permutation of indices
                perm = tf.random.shuffle(tf.range(number_of_nodes))
                # Apply the permutation to both rows and columns
                W_permuted = tf.gather(tf.gather(Ws[i], perm, axis=0), perm, axis=1)
                initializer = ConnectomeInitializer(W_permuted)

            else:
                initializer = random_network_initialization

            rnn_kwargs = {
                'units': number_of_nodes,
                'activation': activation_function,
                'recurrent_initializer': initializer,
                'return_sequences': False,
                'recurrent_regularizer': regu
            }
            if sign_constraint:
                rnn_kwargs['recurrent_constraint'] = NonNeg()

            tf_model = tf.keras.models.Sequential([
                tf.keras.layers.GaussianNoise(stddev=noise_level),
                tf.keras.layers.SimpleRNN(**rnn_kwargs),
                tf.keras.layers.Dense(4, activation='softmax')
            ])
            tf_model.build(input_shape=example_data1[0].shape)
            model[key] = tf_model
        MODELS1.append(model)

In [ ]:
if TASK1:
    for model in tqdm(MODELS1):
        for key, tf_model in model.items():
            tf_model.compile(optimizer=tf.keras.optimizers.Adam(),
                        loss='categorical_crossentropy',
                        metrics=['accuracy'])

            history = tf_model.fit(tfdf1, epochs=10,validation_data=tfdf_test1,
                        callbacks=RNNWeightMatrixHistoryI(RNN_layer_number=1), verbose=0)

            # Metrics
            acc, l0ss, val_acc, val_l0ss, q_stat, smw, degree_assortativity, entropy_weight = model_metrics(history)

            # Save results
            Accuracy1[key].append(acc)
            Loss1[key].append(l0ss)
            Validation_Accuracy1[key].append(val_acc)
            Validation_Loss1[key].append(val_l0ss)
            Modularity1[key].append(q_stat)
            SmallWorldness1[key].append(smw)
            Assortativity1[key].append(degree_assortativity)
            Entropy1[key].append(entropy_weight)

##### **TASK2**

In [ ]:
Accuracy2 = {key: [] for key in regularizers.keys()}
Loss2 = {key: [] for key in regularizers.keys()}
Validation_Accuracy2 = {key: [] for key in regularizers.keys()}
Validation_Loss2 = {key: [] for key in regularizers.keys()}
Modularity2 = {key: [] for key in regularizers.keys()}
SmallWorldness2 = {key: [] for key in regularizers.keys()}
Assortativity2 = {key: [] for key in regularizers.keys()}
Entropy2 = {key: [] for key in regularizers.keys()}

In [ ]:
if TASK2:
    MODELS2 = []
    for i in range(simulations):
        model = {}
        for key, regu in regularizers.items():
            if 'W*' in key:
                initializer = ConnectomeInitializer(Ws[i])

            elif 'W!' in key: #Permutation
                # Generate a random permutation of indices
                perm = tf.random.shuffle(tf.range(number_of_nodes))
                # Apply the permutation to both rows and columns
                W_permuted = tf.gather(tf.gather(Ws[i], perm, axis=0), perm, axis=1)
                initializer = ConnectomeInitializer(W_permuted)

            else:
                initializer = random_network_initialization

            rnn_kwargs = {
                'units': number_of_nodes,
                'activation': activation_function,
                'recurrent_initializer': initializer,
                'return_sequences': False,
                'recurrent_regularizer': regu
            }
            if sign_constraint:
                rnn_kwargs['recurrent_constraint'] = NonNeg()

            tf_model = tf.keras.models.Sequential([
                tf.keras.layers.GaussianNoise(stddev=noise_level),
                tf.keras.layers.SimpleRNN(**rnn_kwargs),
                tf.keras.layers.Dense(2, activation='softmax')
            ])
            tf_model.build(input_shape=example_data2[0].shape)
            model[key] = tf_model
        MODELS2.append(model)

In [ ]:
if TASK2:
    for model in tqdm(MODELS2):
        for key, tf_model in model.items():
            tf_model.compile(optimizer=tf.keras.optimizers.Adam(),
                        loss='categorical_crossentropy',
                        metrics=['accuracy'])

            history = tf_model.fit(tfdf2, epochs=10,validation_data=tfdf_test2,
                        callbacks=RNNWeightMatrixHistoryI(RNN_layer_number=1), verbose=0)

            # Metrics
            acc, l0ss, val_acc, val_l0ss, q_stat, smw, degree_assortativity, entropy_weight = model_metrics(history)

            # Save results
            Accuracy2[key].append(acc)
            Loss2[key].append(l0ss)
            Validation_Accuracy2[key].append(val_acc)
            Validation_Loss2[key].append(val_l0ss)
            Modularity2[key].append(q_stat)
            SmallWorldness2[key].append(smw)
            Assortativity2[key].append(degree_assortativity)
            Entropy2[key].append(entropy_weight)

##### **TASK3**

In [ ]:
Accuracy3 = {key: [] for key in regularizers.keys()}
Loss3 = {key: [] for key in regularizers.keys()}
Validation_Accuracy3 = {key: [] for key in regularizers.keys()}
Validation_Loss3 = {key: [] for key in regularizers.keys()}
Modularity3 = {key: [] for key in regularizers.keys()}
SmallWorldness3 = {key: [] for key in regularizers.keys()}
Assortativity3 = {key: [] for key in regularizers.keys()}
Entropy3 = {key: [] for key in regularizers.keys()}

In [ ]:
if TASK3:
    MODELS3 = []
    for i in range(simulations):
        model = {}
        for key, regu in regularizers.items():
            if 'W*' in key:
                initializer = ConnectomeInitializer(Ws[i])

            elif 'W!' in key: #Permutation
                # Generate a random permutation of indices
                perm = tf.random.shuffle(tf.range(number_of_nodes))
                # Apply the permutation to both rows and columns
                W_permuted = tf.gather(tf.gather(Ws[i], perm, axis=0), perm, axis=1)
                initializer = ConnectomeInitializer(W_permuted)

            else:
                initializer = random_network_initialization

            rnn_kwargs = {
                'units': number_of_nodes,
                'activation': activation_function,
                'recurrent_initializer': initializer,
                'return_sequences': False,
                'recurrent_regularizer': regu
            }
            if sign_constraint:
                rnn_kwargs['recurrent_constraint'] = NonNeg()

            tf_model = tf.keras.models.Sequential([
                tf.keras.layers.GaussianNoise(stddev=noise_level),
                tf.keras.layers.SimpleRNN(**rnn_kwargs),
                tf.keras.layers.Dense(2, activation='softmax')
            ])
            tf_model.build(input_shape=example_data3[0].shape)
            model[key] = tf_model
        MODELS3.append(model)

In [ ]:
if TASK3:
    for model in tqdm(MODELS3):
        for key, tf_model in model.items():
            tf_model.compile(optimizer=tf.keras.optimizers.Adam(),
                        loss='categorical_crossentropy',
                        metrics=['accuracy'])

            history = tf_model.fit(tfdf3, epochs=10,validation_data=tfdf_test3,
                        callbacks=RNNWeightMatrixHistoryI(RNN_layer_number=1), verbose=0)

            # Metrics
            acc, l0ss, val_acc, val_l0ss, q_stat, smw, degree_assortativity, entropy_weight = model_metrics(history)

            # Save results
            Accuracy3[key].append(acc)
            Loss3[key].append(l0ss)
            Validation_Accuracy3[key].append(val_acc)
            Validation_Loss3[key].append(val_l0ss)
            Modularity3[key].append(q_stat)
            SmallWorldness3[key].append(smw)
            Assortativity3[key].append(degree_assortativity)
            Entropy3[key].append(entropy_weight)

#### Save

In [ ]:
os.makedirs(save_folder_name)

# SAVE config.txt
with open(f"{save_folder_name}/config.txt", "w") as f:
    f.write(f"session = {session}\n")
    f.write(f"scan = {scan}\n")
    f.write(f"field = {field}\n")
    f.write(f"simulations = {simulations}\n")
    f.write(f"number_of_nodes = {number_of_nodes}\n")
    f.write(f"network_grid = {network_grid}\n")
    f.write(f"sign_constraint = {sign_constraint}\n")
    f.write(f"use_only_sttc = {use_only_sttc}\n")
    f.write(f"use_precison_matrix = {use_precison_matrix}\n")
    f.write(f"noise_level = {noise_level}\n")
    f.write(f"regu_strength = {regu_strength}\n")
    f.write(f"emd_strength = {emd_strength}\n")
    f.write(f"activation_function = {activation_function}\n")
    f.write(f"random_network_initialization = {random_network_initialization}\n")

In [ ]:
# Define tasks and metrics
tasks, task_binary = ['Task1', 'Task2', 'Task3'], [TASK1, TASK2, TASK3]
metrics = ['Accuracy', 'Loss', 'Validation_Accuracy', 'Validation_Loss', 'Modularity', 'SmallWorldness', 'Assortativity', 'Entropy']
regularizer_keys = list(regularizers.keys())

# Save raw data for each task and metric combination
for task_idx, task in enumerate(tasks):
    if task_binary[tasks.index(task)]:
        task_folder = f'{save_folder_name}/{task}'
        os.makedirs(task_folder, exist_ok=True)

        for metric in metrics:
            # Get the raw data for this metric and task
            raw_data = eval(f'{metric}{task_idx+1}')

            # Create DataFrame with raw simulation data
            raw_df = pd.DataFrame(raw_data)
            raw_df.index.name = 'simulation'

            # Save raw data to CSV
            raw_df.to_csv(f'{task_folder}/{metric}_raw.csv', index=True)

# Also create a summary file with means for quick reference
metrics_data = {}
for task in tasks:
    if task_binary[tasks.index(task)]:
        metrics_data[task] = {}
        for metric in metrics:
            metrics_data[task][metric] = {}
            for key in regularizer_keys:
                metrics_data[task][metric][key] = mean(eval(f'{metric}{tasks.index(task)+1}["{key}"]'))

# Convert the dictionary to a multi-index DataFrame
metrics_df = pd.DataFrame.from_dict({(i, j): metrics_data[i][j]
                                     for i in metrics_data.keys()
                                     for j in metrics_data[i].keys()},
                                    orient='index')

# Set the column names
metrics_df.columns = regularizer_keys

# Reset the index to ensure the header is correct
metrics_df.reset_index(inplace=True)

# Rename the columns to ensure the header is correct
metrics_df.columns = ['Task', 'Metric'] + regularizer_keys

# Save the summary DataFrame to a CSV file
metrics_df.to_csv(f'{save_folder_name}/metrics.csv', index=False)

In [ ]:
# Also create a summary file with standard deviations for quick reference
metrics_SD = ['Accuracy_SD', 'Loss_SD', 'Validation_Accuracy_SD', 'Validation_Loss_SD', 'Modularity_SD', 'SmallWorldness_SD', 'Assortativity_SD', 'Entropy_SD']

# Create a dictionary with the metrics
metrics_data_SD = {}
for task in tasks:
    if task_binary[tasks.index(task)]:
        metrics_data_SD[task] = {}
        for metric in metrics_SD:
            metrics_data_SD[task][metric] = {}
            for key in regularizer_keys:
                metrics_data_SD[task][metric][key] = (
        (lambda v: stdev([x for x in v if not math.isnan(x)]) if isinstance(v, (list, tuple)) and len([x for x in v if not math.isnan(x)]) > 1 else 0.0)
        (eval(f'{metric[:-3]}{tasks.index(task)+1}["{key}"]'))
    )

# Convert the dictionary to a multi-index DataFrame
metrics_SD_df = pd.DataFrame.from_dict({(i, j): metrics_data_SD[i][j]
                                     for i in metrics_data_SD.keys()
                                     for j in metrics_data_SD[i].keys()},
                                    orient='index')

# Set the column names
metrics_SD_df.columns = regularizer_keys

# Reset the index to ensure the header is correct
metrics_SD_df.reset_index(inplace=True)

# Rename the columns to ensure the header is correct
metrics_SD_df.columns = ['Task', 'Metric'] + regularizer_keys

# Save the DataFrame to a CSV file
metrics_SD_df.to_csv(f'{save_folder_name}/metrics_SD.csv', index=False)

import shutil
shutil.make_archive(save_folder_name, 'zip', save_folder_name)


In [ ]:
from google.colab import files
files.download(f'{save_folder_name}.zip')